# 34 · Disaggregated prefill and multi-node serving
Level L5 · Part 5 · core · ~20 min · builds on 23, 30

## Goal
By the end of this lesson you can draw the prefill and decode split across two GPU pools, say what moves between them and how big it is for a Brightside request (the support-desk assistant from lesson 01), and explain why the tensor-parallel rule is per node: tensor parallelism stays inside one node, and pipeline parallelism is what crosses nodes.

## Why it matters
- **Without it:** one pool of GPUs does both jobs. A long prompt keeps the GPUs busy in one big prefill burst, and every other user's decode steps wait behind it. The answer stream stalls. The median latency looks fine, but the p99 (the slowest 1% of tokens) inter-token latency (ITL, the wait between two streamed tokens) spikes.
- **With it:** a prefill pool reads the prompt, the KV cache moves once to a decode pool, and the decode pool streams tokens with no prefill in its way. You size the two pools separately: the prefill pool sets time to first token (TTFT), the decode pool sets ITL.
- **When you don't need it:** short prompts on one node. The KV transfer and a second fleet to run cost more than the stalls they remove. Lesson 30 shows how a single pool can still be routed well.

## The idea
**Disaggregated serving** (prefill/decode disaggregation, or P/D) runs the two phases on different GPU groups. A **prefill** worker reads the whole prompt and builds its KV cache (lesson 02). It sends that cache to a **decode** worker, which generates the answer one token per step (lesson 01). The router sends each request to a prefill worker and then to a decode worker.

Think of a restaurant with a prep station and a grill. The prep cooks chop and portion the ingredients once per order. Then the ingredients go to the grill, which cooks the plate one step at a time, and no prep work lands on the grill in the middle of service.

Inside each group the old rule still holds (lessons 22 and 23). Tensor parallelism cuts every layer across GPUs that must join their results after every layer. Pipeline parallelism cuts the model into stages, one per node, and passes one small activation between them.

## Picture
One Brightside request, split across two pools:
```mermaid
flowchart LR
  R["request: 1,600-token prompt"] --> P["prefill pool: one parallel pass over the prompt"]
  P -->|"KV cache for 1,600 tokens, moved once"| D["decode pool: one new token per step"]
  D --> O["answer tokens stream out"]
```

## Explained
**What moves.** After prefill, the prompt's keys and values sit in the prefill GPUs' memory. For the 70B model in FP16 that is 320 KiB per token, summed over all its GPUs (lesson 23). The 1,600-token Brightside prompt is about 500 MiB, and it moves once per request. The decode pool then appends one token's keys and values per step, locally.

**Why separate the phases.** Prefill processes many prompt tokens at once, so it is compute-bound (lesson 06). Decode reads the weights for every token and is memory-bound. A prefill burst on the same GPUs as the decode steps delays those steps for everybody. Two pools let you tune each one: add prefill GPUs for TTFT, add decode GPUs for ITL.


**What it costs.** The cache moves over the network, and the decode pool needs room for it. Both pools must run the same model and cache format, and the cache is moved in blocks (the **block format** is how the cache is cut into fixed-size chunks of tokens). Meta's stack uses blocks of 128 or 256 tokens instead of vLLM's default of 16. Bigger blocks mean fewer, larger transfers. The router must also track both pools, which is why Cloudflare's load balancer counts in-flight prefill and decode tokens per endpoint.

**The status.** vLLM still labels disaggregated prefill "experimental and subject to change". Its docs say it **does not improve throughput**. It lets you tune TTFT and ITL separately, and it helps control the tail ITL. Gains quoted by vendors are their own measurements.


**The tensor-parallel rule.** vLLM's parallelism docs say to set `tensor_parallel_size` to the number of GPUs per node, and `pipeline_parallel_size` to the number of nodes. The reason is the cost of each call. A 70B model runs 160 all-reduces for every forward pass (2 per layer, 80 layers). An **all-reduce** is one collective call: every GPU in the group holds a partial result, and every GPU ends with the total. NCCL (NVIDIA's GPU communication library) runs these calls. Each all-reduce waits for every GPU in its group. Inside a node, the link is fast and the wait is short. Across a network, each of those 160 calls pays the network's latency. A pipeline boundary, by contrast, sends one 16 KiB activation per token, so it tolerates a slower link.

## Worked example: the KV bytes and the all-reduce calls
The first cell prices the KV transfer for a few prompt lengths on two links. The second prices the all-reduces of one decode token. The link speeds and per-call latencies are labelled assumptions; the model sizes come from lesson 23. PCIe applies only when both pools share a node; the network applies across nodes. The network column is a 100 GbE link (100 gigabits per second, 12.5 GB/s).

In [1]:
MIB, GIB = 2**20, 2**30
KV_PER_TOKEN = 320 * 1024          # 70B FP16 cache, all GPUs together (lesson 23)
PCIE = 32e9                        # PCIe Gen4 x16, one direction, bytes per second (lesson 22)
NET = 100e9 / 8                    # ASSUMPTION: a 100 GbE network link (100 Gb/s), 12.5 GB/s

def transfer_ms(prompt_tokens, link):
    """Time to move one request's KV cache from the prefill pool to the decode pool."""
    return prompt_tokens * KV_PER_TOKEN / link * 1e3

print(f"{'prompt':>8} {'KV moved':>10} {'PCIe ms':>9} {'100 GbE ms':>11}")
for prompt in (200, 1_600, 8_000, 32_000):
    kv_mib = prompt * KV_PER_TOKEN / MIB
    print(f"{prompt:>8,} {kv_mib:>6,.0f} MiB {transfer_ms(prompt, PCIE):>9.1f} {transfer_ms(prompt, NET):>11.1f}")

  prompt   KV moved   PCIe ms  100 GbE ms
     200     62 MiB       2.0         5.2
   1,600    500 MiB      16.4        41.9
   8,000  2,500 MiB      81.9       209.7
  32,000 10,000 MiB     327.7       838.9


**What you should see:** the Brightside 1,600-token prompt moves 500 MiB, about 16 ms over PCIe and about 42 ms over the assumed 100 GbE link. A 32,000-token document moves 10,000 MiB, so the transfer grows with the prompt, while the decode steps do not wait for a prefill. The transfer happens once per request, not once per token.

**Before the formula.** A ring all-reduce of M bytes over n GPUs sends each GPU a 2(n−1)/n share of the data (lesson 22 derives it), so the bytes per call barely grow as the group grows. Each call also pays a fixed latency however small its message is, which is why 160 small calls cost more than their bytes suggest. The weight-read line below (58.8 ms per token) is the comparison baseline: the all-reduce time is only acceptable if it stays small next to it.

In [2]:
PARAMS_70B = 70_553_706_496
BW = 300e9                         # L4 memory bandwidth (lessons 06 and 23)
ACT = 8192 * 2                     # one token's activation vector, 16 KiB (lesson 22)
ALLREDUCES = 160                   # per forward pass: 2 per layer, 80 layers

def decode_comm_ms(tp, link, fixed_us):
    """All-reduce time of one decode token: 160 ring all-reduces, each with a fixed per-call latency."""
    per_call = 2 * (tp - 1) / tp * ACT / link + fixed_us * 1e-6
    return ALLREDUCES * per_call * 1e3

read_ms = PARAMS_70B * 2 / (8 * BW) * 1e3
print(f"TP=8, weight reads per token on one node (L4, as in lessons 06 and 23): {read_ms:5.1f} ms")
print(f"TP=8, all-reduces on PCIe (20 us per call):    {decode_comm_ms(8, PCIE, 20):5.1f} ms")
print(f"TP=8, all-reduces over the network (100 us per call, ASSUMPTION): {decode_comm_ms(8, NET, 100):5.1f} ms")

TP=8, weight reads per token on one node (L4, as in lessons 06 and 23):  58.8 ms
TP=8, all-reduces on PCIe (20 us per call):      3.3 ms
TP=8, all-reduces over the network (100 us per call, ASSUMPTION):  16.4 ms


**What you should see:** on one node, TP=8 reads the weights for about 59 ms per token and spends about 3 ms on all-reduces. The same TP=8 split over a network, at an assumed 100 µs per call, spends about 16 ms per token on all-reduces alone, and it pays that on every token. That is why the rule puts tensor parallelism inside a node and uses pipeline parallelism across nodes.

## At work
- **Cloudflare (May 2026)** reports that prefill/decode disaggregation cut its p90 (the 90th percentile) inter-token latency from about 100 ms, with high variance, to 20 to 30 ms, about 3 times lower. Its load balancer tracks in-flight prefill and decode tokens per endpoint. → lesson 30 (routing across replicas and pools).
- **Meta and PyTorch (Sep 2025)** run P/D disaggregation with vLLM. Sticky routing and load balancing gave a 40 to 50% prefix cache hit at 90% HBM (the GPU's own memory) use. TTFT regressed sharply under heavy load over TCP, a network bottleneck. → lesson 14 (prefix caching: the cache that sticky routing keeps hitting) and lesson 33 (failure drills, a later lesson: what happens when a link or a replica fails).
- **vLLM's docs** call disaggregation experimental and say it does not improve throughput. Teams that adopt it measure TTFT and ITL on their own traffic first. → lesson 27 (load tests that match reality).


**Sources:** Cloudflare, "High-performance LLMs": https://blog.cloudflare.com/high-performance-llms/ · Meta and PyTorch, "Disaggregated Inference at Scale with PyTorch & vLLM": https://pytorch.org/blog/disaggregated-inference-at-scale-with-pytorch-vllm/ · vLLM, "Disaggregated prefilling": https://docs.vllm.ai/en/latest/features/disagg_prefill/ · vLLM, "Parallelism and scaling": https://docs.vllm.ai/en/latest/serving/parallelism_scaling/

## Common misconceptions
1. **"Disaggregation makes the server faster."** vLLM's docs say it does not improve throughput. It separates the phases so that you can tune TTFT and the tail ITL on their own. Vendor gains come from their own setups and loads.
2. **"Use tensor parallelism across nodes to fit a bigger model."** Each token pays 160 all-reduce calls, and each call waits on the slowest link in its group. Put tensor parallelism inside a node, and pipeline parallelism across nodes.
3. **"Moving the cache is free, it is just memory."** A 1,600-token Brightside request moves 500 MiB, about 16 ms over PCIe and about 42 ms over the assumed network. The decode pool also needs room for the cache it receives.

## Production checklist
- [ ] Measure TTFT and p99 ITL on one pool and on two, with your own traffic shape (lesson 27), before you keep the split.
- [ ] Compute the transfer time for your longest prompt: prompt tokens × KV bytes per token ÷ link speed. Add it to the TTFT budget.
- [ ] Run the same model, KV dtype and block format on both pools. A mismatch fails at transfer time.
- [ ] Set tensor parallelism to the GPUs per node and pipeline parallelism to the node count. Check the NCCL log (NVIDIA's GPU communication library) for `NET/IB` (InfiniBand, the fast network) rather than `NET/Socket` (plain TCP).
- [ ] Route by pool: count in-flight prefill and decode tokens per endpoint (lesson 30).
- [ ] Alert on decode-pool KV cache use and on the queue in front of the prefill pool.
- [ ] Keep the single-pool setup ready to roll back, and switch traffic over gradually, comparing TTFT and p99 ITL against the single pool at each step.

## Check yourself
1. A 32,000-token document arrives while others stream answers. What happens on one pool, and what does splitting change?
2. Why does the KV cache move once per request, not once per token?
3. A 70B model runs on two 8-GPU nodes. Which `tensor_parallel_size` and `pipeline_parallel_size` do you set, and why?
4. Throughput is unchanged after you split the pools. Is the setup broken? Which numbers should you check?
5. How long does the 1,600-token Brightside cache take over the 100 GbE link? Use the first worked cell.
6. Predict the 4,000-token transfer over 100 GbE, then 25 GbE (`NET = 25e9 / 8`). Add 4,000 to the first cell's list to check.

**Answers**
1. On one pool, the streaming requests stall, because their decode steps wait behind the 32,000-token prefill. With two pools, the decode pool never runs that prefill, so their ITL stays flat. The document pays its own TTFT in the prefill pool.
2. Prefill builds the cache for the prompt once. Each decode step adds one token's keys and values on the decode side, so only the prompt's cache has to cross the network.

3. `tensor_parallel_size=8` (GPUs per node) and `pipeline_parallel_size=2` (nodes). The 160 all-reduces per token stay on the fast link inside each node, and the boundary between nodes carries one 16 KiB activation per token.
4. Not necessarily. vLLM's docs say disaggregation does not improve throughput. Check TTFT and the p99 ITL on your own traffic, against the same setup without it.
5. About 42 ms (500 MiB divided by 12.5 GB/s, as the first cell prints). Add that to TTFT when you check the budget.
6. A 4,000-token prompt is about 1,250 MiB, so about 105 ms over 100 GbE (1.31 GB ÷ 12.5 GB/s). Over 25 GbE (3.125 GB/s) the same transfer takes about 419 ms, four times longer, because the time is the bytes divided by the link speed.

## Recap
- **Disaggregated serving** runs prefill and decode on separate pools. The prefill pool builds the KV cache, and the decode pool streams the answer.
- The **cache transfer** is paid once per request: 320 KiB per token for 70B in FP16, so 500 MiB for the 1,600-token Brightside prompt.
- It is for **tail latency and separate tuning**, not for throughput. vLLM still labels it experimental.
- **Tensor parallelism stays inside a node** (160 all-reduces per token). **Pipeline parallelism crosses nodes** (one small activation per boundary).

**Interview:** "How would you serve a 70B model on two 8-GPU nodes, and when would you split prefill from decode?" Strong answer: tensor parallel 8 inside each node, pipeline parallel 2 across them, because the all-reduces need the fast link. Split prefill from decode only when long prompts and a strict ITL target coexist. Check the KV transfer time against the TTFT budget, and measure TTFT and ITL, not throughput.